In [1]:
!nvidia-smi

Mon Apr 20 10:54:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip -q install transformers accelerate sentencepiece

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.10.0+cu128
True
Tesla T4


In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model = model.to("cuda")
model.eval()

print("Loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded successfully


In [5]:
prompt = "Explain the role of attention in transformer models."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        use_cache=True
    )

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Explain the role of attention in transformer models. In a transformer model, attention is used to focus on relevant parts of the input data and improve the performance of the model. Attention mechanisms are typically applied at the


In [14]:
%%writefile benchmark_concurrency_skeleton.py
# benchmark_day5_concurrency_skeleton.py
# Day 5 - Multi-concurrency benchmark skeleton
# Goal:
#   1. Sweep concurrency = 1 / 2 / 4 / 8 / 16
#   2. Measure throughput / avg latency / p95 latency
#   3. Optionally collect GPU utilization
#   4. Plot throughput vs concurrency
#   5. Plot latency vs concurrency
#
# NOTE:
#   Core logic is intentionally left as TODO.

import time
import json
import math
import statistics
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

import torch
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed

from transformers import AutoTokenizer, AutoModelForCausalLM

# TODO:
# import HF classes you need
# from transformers import ...
#
# Optional:
# if you want GPU utilization:
# import pynvml


# =========================
# Config
# =========================

@dataclass
class ConcurrencyConfig:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    dtype: str = "float16"

    # concurrency sweep
    concurrency_levels: List[int] = None

    # workload
    fixed_prompt_len: int = 128
    fixed_output_len: int = 64

    # number of requests per concurrency setting
    # e.g. if concurrency=4 and num_requests_per_level=20,
    # total 20 requests are launched with at most 4 in flight.
    num_requests_per_level: int = 20

    # warmup
    num_warmup: int = 1

    # generation behavior
    do_sample: bool = False
    temperature: float = 1.0
    top_p: float = 1.0
    use_kv_cache: bool = True

    # output
    results_dir: str = "./results_day5"

    def __post_init__(self):
        if self.concurrency_levels is None:
            self.concurrency_levels = [1, 2, 4, 8, 16]


# =========================
# Basic utils
# =========================

def ensure_dir(path: str):
    Path(path).mkdir(parents=True, exist_ok=True)


def resolve_torch_dtype(dtype_str: str):
    mapping = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    if dtype_str not in mapping:
        raise ValueError(f"Unsupported dtype: {dtype_str}")
    return mapping[dtype_str]


def save_json(obj, out_path: str):
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def percentile(values: List[float], p: float) -> float:
    """
    Return percentile p (e.g. 95) for a list of numbers.
    """
    # TODO:
    # implement percentile calculation
    # raise NotImplementedError
    if not values:
        return 0.0
    xs = sorted(values)
    if len(xs) == 1:
        return xs[0]

    k = (len(xs) - 1) * (p / 100.0)
    f = math.floor(k)
    c = math.ceil(k)

    if f == c:
        return xs[int(k)]

    d0 = xs[f] * (c - k)
    d1 = xs[c] * (k - f)
    return d0 + d1


# =========================
# Prompt helpers
# =========================

def build_base_prompt() -> str:
    """
    Return a reusable base prompt.
    """
    # TODO:
    # reuse your Day 3 / Day 4 version
    # raise NotImplementedError
    return (
        "You are a helpful AI assistant. "
        "Please explain the role of attention in transformer models, "
        "including why attention is useful for long-range dependency modeling. "
    )


def make_prompt_near_target_tokens(
    tokenizer,
    base_text: str,
    target_prompt_len: int,
    max_iters: int = 100,
) -> str:
    """
    Build a prompt whose tokenized length is close to target_prompt_len.
    """
    # TODO:
    # reuse your Day 3 / Day 4 version
    # raise NotImplementedError
    text = base_text
    filler = (
        "Add more explanation about self-attention, key-value interactions, "
        "and contextual representation learning. "
    )

    for _ in range(max_iters):
        ids = tokenizer(text, return_tensors="pt").input_ids[0]
        if ids.shape[0] >= target_prompt_len:
            ids = ids[:target_prompt_len]
            text = tokenizer.decode(ids, skip_special_tokens=True)
            return text
        text += filler

    ids = tokenizer(text, return_tensors="pt").input_ids[0][:target_prompt_len]
    return tokenizer.decode(ids, skip_special_tokens=True)


def count_input_tokens(tokenizer, prompt: str) -> int:
    """
    Count input tokens.
    """
    # TODO
    # raise NotImplementedError
    return tokenizer(prompt, return_tensors="pt").input_ids.shape[1]


# =========================
# Model loading
# =========================

def load_tokenizer_and_model(cfg: ConcurrencyConfig):
    torch_dtype = resolve_torch_dtype(cfg.dtype)

    # TODO:
    # 1. load tokenizer
    # 2. load model
    # 3. set pad token if needed
    # 4. move model to device
    # 5. model.eval()
    # raise NotImplementedError
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        cfg.model_name,
        torch_dtype=torch_dtype,
    )

    model = model.to(cfg.device)
    model.eval()
    return tokenizer, model


def prepare_inputs(tokenizer, prompt: str, device: str):
    """
    Tokenize prompt and move to device.
    """
    # TODO
    # raise NotImplementedError
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    return inputs


# =========================
# Optional GPU util monitor
# =========================

class GPUMonitor:
    """
    Optional helper for sampling GPU utilization while a concurrency test runs.

    You can implement this with pynvml if available.
    Suggested behavior:
      - start() begins sampling in background
      - stop() ends sampling
      - summary() returns avg/max utilization
    """
    def __init__(self, enabled: bool = False, device_index: int = 0, sample_interval_sec: float = 0.2):
        self.enabled = enabled
        self.device_index = device_index
        self.sample_interval_sec = sample_interval_sec

        # TODO:
        # initialize fields for background thread / samples
        # optionally initialize NVML
        pass

    def start(self):
        # TODO
        pass

    def stop(self):
        # TODO
        pass

    def summary(self) -> Dict[str, Any]:
        # TODO:
        # return something like:
        # {
        #   "gpu_util_avg": ...,
        #   "gpu_util_max": ...,
        #   "mem_util_avg": ...,
        # }
        return {}


# =========================
# Single request run
# =========================

def run_single_request(
    model,
    tokenizer,
    prompt: str,
    cfg: ConcurrencyConfig,
    request_id: int,
) -> Dict[str, Any]:
    """
    Run one request and record per-request metrics.

    At minimum record:
      - request_id
      - input_tokens
      - generated_tokens
      - latency_sec
      - output_tokens_per_sec

    This is the basic unit used by the concurrency harness.
    """
    inputs = prepare_inputs(tokenizer, prompt, cfg.device)

    if cfg.device == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        # TODO:
        # call model.generate(...) or your manual decode logic
        if cfg.do_sample:
                outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.fixed_output_len,
                do_sample=True,
                temperature=cfg.temperature,
                top_p=cfg.top_p,
                use_cache=cfg.use_kv_cache,
                pad_token_id=tokenizer.pad_token_id,
            )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=cfg.fixed_output_len,
                do_sample=False,
                use_cache=cfg.use_kv_cache,
                pad_token_id=tokenizer.pad_token_id,
            )

    if cfg.device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    latency_sec = end - start

    # TODO:
    # 1. compute input_tokens
    # 2. compute generated_tokens
    # 3. compute output_tokens_per_sec
    # 4. optionally decode text
    # 5. return dict
    # raise NotImplementedError

    input_tokens = inputs["input_ids"].shape[1]
    total_output_tokens = outputs.shape[1]
    generated_tokens = max(total_output_tokens - input_tokens, 0)

    output_tokens_per_sec = (
        generated_tokens / latency_sec if latency_sec > 0 else 0.0
    )

    return {
        "request_id": request_id,
        "input_tokens": int(input_tokens),
        "generated_tokens": int(generated_tokens),
        "latency_sec": float(latency_sec),
        "output_tokens_per_sec": float(output_tokens_per_sec),
    }


# =========================
# Warmup
# =========================

def run_warmup(
    model,
    tokenizer,
    prompt: str,
    cfg: ConcurrencyConfig,
):
    for i in range(cfg.num_warmup):
        print(f"[Warmup {i+1}/{cfg.num_warmup}]")
        _ = run_single_request(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            cfg=cfg,
            request_id=-1,
        )


# =========================
# Concurrency benchmark core
# =========================

def run_one_concurrency_level(
    model,
    tokenizer,
    prompt: str,
    cfg: ConcurrencyConfig,
    concurrency: int,
    gpu_monitor: Optional[GPUMonitor] = None,
) -> Dict[str, Any]:
    """
    Benchmark one concurrency level.

    Suggested approach:
      - launch num_requests_per_level requests
      - allow at most `concurrency` workers in flight
      - collect per-request latency
      - compute total wall-clock time
      - derive throughput and latency metrics

    Throughput choices:
      A) requests/sec = total_requests / wall_time
      B) tokens/sec   = total_generated_tokens / wall_time

    You can record both if you want.
    """
    total_requests = cfg.num_requests_per_level

    per_request_records: List[Dict[str, Any]] = []

    if gpu_monitor is not None:
        gpu_monitor.start()

    wall_start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = []
        for request_id in range(total_requests):
            future = executor.submit(
                run_single_request,
                model,
                tokenizer,
                prompt,
                cfg,
                request_id,
            )
            futures.append(future)

    for future in as_completed(futures):
        record = future.result()
        per_request_records.append(record)

    # TODO:
    # Use ThreadPoolExecutor(max_workers=concurrency)
    # Submit total_requests tasks:
    #
    #   future = executor.submit(run_single_request, ...)
    #
    # Then collect each finished result with as_completed(...)
    #
    # Append per_request_records
    #
    # Note:
    # this is a simple concurrency harness, not a real production scheduler.
    # raise NotImplementedError

    wall_end = time.perf_counter()

    if gpu_monitor is not None:
        gpu_monitor.stop()

    wall_time_sec = wall_end - wall_start

    # TODO:
    # Compute:
    # - avg_latency_sec
    # - p95_latency_sec
    # - requests_per_sec
    # - tokens_per_sec
    # - total_generated_tokens
    #
    # Also optionally merge gpu_monitor.summary()
    #
    # Return:
    # {
    #   "concurrency": ...,
    #   "total_requests": ...,
    #   "wall_time_sec": ...,
    #   "avg_latency_sec": ...,
    #   "p95_latency_sec": ...,
    #   "requests_per_sec": ...,
    #   "tokens_per_sec": ...,
    #   "per_request_records": [...],
    #   ...
    # }
    # raise NotImplementedError
    latencies = [r["latency_sec"] for r in per_request_records]
    total_generated_tokens = sum(r["generated_tokens"] for r in per_request_records)

    avg_latency_sec = sum(latencies) / len(latencies) if latencies else 0.0
    p95_latency_sec = percentile(latencies, 95)

    requests_per_sec = (
        total_requests / wall_time_sec if wall_time_sec > 0 else 0.0
    )
    tokens_per_sec = (
        total_generated_tokens / wall_time_sec if wall_time_sec > 0 else 0.0
    )

    summary = {
        "concurrency": concurrency,
        "total_requests": total_requests,
        "wall_time_sec": float(wall_time_sec),
        "avg_latency_sec": float(avg_latency_sec),
        "p95_latency_sec": float(p95_latency_sec),
        "requests_per_sec": float(requests_per_sec),
        "tokens_per_sec": float(tokens_per_sec),
        "total_generated_tokens": int(total_generated_tokens),
        "per_request_records": per_request_records,
    }
    return summary


def run_concurrency_sweep(
    model,
    tokenizer,
    prompt: str,
    cfg: ConcurrencyConfig,
) -> List[Dict[str, Any]]:
    """
    Sweep all concurrency levels.
    """
    summaries = []

    for c in cfg.concurrency_levels:
        print(f"\n=== Concurrency Level: {c} ===")

        # Optional:
        # gpu_monitor = GPUMonitor(enabled=True)
        # or disabled if you don't want to implement it yet
        gpu_monitor = GPUMonitor(enabled=False)

        summary = run_one_concurrency_level(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            cfg=cfg,
            concurrency=c,
            gpu_monitor=gpu_monitor,
        )

        summaries.append(summary)

        print(json.dumps({
            "concurrency": summary.get("concurrency"),
            "avg_latency_sec": round(summary.get("avg_latency_sec", -1), 4),
            "p95_latency_sec": round(summary.get("p95_latency_sec", -1), 4),
            "requests_per_sec": round(summary.get("requests_per_sec", -1), 4),
            "tokens_per_sec": round(summary.get("tokens_per_sec", -1), 4),
        }, indent=2))

    return summaries


# =========================
# Plotting
# =========================

def plot_throughput_vs_concurrency(
    sweep_summaries: List[Dict[str, Any]],
    out_path: str,
    throughput_key: str = "tokens_per_sec",
):
    """
    Plot throughput vs concurrency.
    throughput_key can be:
      - "tokens_per_sec"
      - "requests_per_sec"
    """
    # TODO:
    # 1. extract x and y
    # 2. create matplotlib plot
    # 3. save figure
    # 4. optionally plt.show()
    # raise NotImplementedError
    xs = [d["concurrency"] for d in sweep_summaries]
    ys = [d[throughput_key] for d in sweep_summaries]

    plt.figure(figsize=(7, 5))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("Concurrency")
    plt.ylabel(throughput_key)
    plt.title(f"{throughput_key} vs Concurrency")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()



def plot_latency_vs_concurrency(
    sweep_summaries: List[Dict[str, Any]],
    out_path: str,
    latency_key: str = "avg_latency_sec",
):
    """
    Plot latency vs concurrency.
    latency_key can be:
      - "avg_latency_sec"
      - "p95_latency_sec"
    """
    # TODO:
    # 1. extract x and y
    # 2. create matplotlib plot
    # 3. save figure
    # 4. optionally plt.show()
    # raise NotImplementedError
    xs = [d["concurrency"] for d in sweep_summaries]
    ys = [d[latency_key] for d in sweep_summaries]

    plt.figure(figsize=(7, 5))
    plt.plot(xs, ys, marker="o")
    plt.xlabel("Concurrency")
    plt.ylabel(latency_key)
    plt.title(f"{latency_key} vs Concurrency")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()

# =========================
# Optional combined plot helpers
# =========================

def plot_avg_and_p95_latency_vs_concurrency(
    sweep_summaries: List[Dict[str, Any]],
    out_path: str,
):
    """
    Optional helper: plot avg latency and p95 latency together.
    """
    # TODO
    # raise NotImplementedError
    xs = [d["concurrency"] for d in sweep_summaries]
    avg_ys = [d["avg_latency_sec"] for d in sweep_summaries]
    p95_ys = [d["p95_latency_sec"] for d in sweep_summaries]

    plt.figure(figsize=(7, 5))
    plt.plot(xs, avg_ys, marker="o", label="avg latency")
    plt.plot(xs, p95_ys, marker="s", label="p95 latency")
    plt.xlabel("Concurrency")
    plt.ylabel("Latency (sec)")
    plt.title("Average and P95 Latency vs Concurrency")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path)
    plt.show()
    plt.close()


# =========================
# Main
# =========================

def main():
    cfg = ConcurrencyConfig()

    print("=== Day 5 Concurrency Config ===")
    print(cfg)

    ensure_dir(cfg.results_dir)

    tokenizer, model = load_tokenizer_and_model(cfg)

    base_prompt = build_base_prompt()
    prompt = make_prompt_near_target_tokens(
        tokenizer=tokenizer,
        base_text=base_prompt,
        target_prompt_len=cfg.fixed_prompt_len,
    )

    actual_input_tokens = count_input_tokens(tokenizer, prompt)
    print(f"Actual input tokens: {actual_input_tokens}")

    print("\n=== Warmup ===")
    run_warmup(model, tokenizer, prompt, cfg)

    print("\n=== Concurrency Sweep ===")
    sweep_summaries = run_concurrency_sweep(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        cfg=cfg,
    )

    summary_path = str(Path(cfg.results_dir) / "concurrency_sweep_summary.json")
    save_json(sweep_summaries, summary_path)

    plot_throughput_vs_concurrency(
        sweep_summaries=sweep_summaries,
        out_path=str(Path(cfg.results_dir) / "throughput_vs_concurrency.png"),
        throughput_key="tokens_per_sec",
    )

    plot_latency_vs_concurrency(
        sweep_summaries=sweep_summaries,
        out_path=str(Path(cfg.results_dir) / "avg_latency_vs_concurrency.png"),
        latency_key="avg_latency_sec",
    )

    # optional:
    # plot_avg_and_p95_latency_vs_concurrency(
    #     sweep_summaries=sweep_summaries,
    #     out_path=str(Path(cfg.results_dir) / "avg_p95_latency_vs_concurrency.png"),
    # )

    print("\nSaved:")
    print(summary_path)
    print(str(Path(cfg.results_dir) / "throughput_vs_concurrency.png"))
    print(str(Path(cfg.results_dir) / "avg_latency_vs_concurrency.png"))


if __name__ == "__main__":
    main()

Overwriting benchmark_concurrency_skeleton.py


In [15]:
!python benchmark_concurrency_skeleton.py

=== Day 5 Concurrency Config ===
ConcurrencyConfig(model_name='Qwen/Qwen2.5-0.5B-Instruct', device='cuda', dtype='float16', concurrency_levels=[1, 2, 4, 8, 16], fixed_prompt_len=128, fixed_output_len=64, num_requests_per_level=20, num_warmup=1, do_sample=False, temperature=1.0, top_p=1.0, use_kv_cache=True, results_dir='./results_day5')
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:00<00:00, 343.58it/s, Materializing param=model.norm.weight]
Actual input tokens: 128

=== Warmup ===
[Warmup 1/1]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.

=== Concurrency Sweep ===

=== Concurrency Level: 1 ===
{
  "concurrency": 1,
  "avg_latency_sec": 2.4989,
  "p95_latency_sec": 2.9201,
  "requests_per_sec": 0.4,
  "tokens_per_sec": 25.6004
}

=== Concurrency Level: 2 ===
{
  "concurrency": 2,
  "avg_latency_sec": 5.4369,
  "p95_latency_sec": 6.7935,
  "req